# 01. Building the raw dataset from NASA POWER

This notebook downloads the raw data used by the project and only needs to be run once.
The main analysis is in `02_rain_prediction.ipynb`.

**Source.** [NASA POWER](https://power.larc.nasa.gov/docs/services/api/temporal/daily/)
(Prediction Of Worldwide Energy Resources), a free NASA service that provides daily
meteorological and solar values derived from satellite observations and the MERRA-2
reanalysis. The *AG* (agroclimatology) community is requested.

**Locations.** Twelve district centres spread across Rwanda's agro-ecological zones
(Kigali, Musanze, Rubavu, Huye, Rusizi, Nyagatare, Kirehe, Muhanga, Nyamagabe, Rwamagana,
Karongi and Gicumbi), daily from 1 January 1981 to 31 December 2025.

**Parameters (14).** Corrected precipitation (`PRECTOTCORR`), mean, maximum and minimum
temperature at 2 m, dew point, relative and specific humidity, surface pressure, mean
and maximum wind speed and wind direction at 2 m, all-sky surface shortwave radiation,
cloud amount and top-soil wetness.

**Why some centres are removed.** NASA POWER does not return a value at the exact
coordinates but for the grid cell (0.5 x 0.625 degrees) that contains them. Two centres
inside the same cell receive identical series. Keeping both would count the same weather
twice, and a model could learn a day from one copy and be tested on its twin. Each
location's rainfall series is therefore fingerprinted and duplicates are dropped
(Muhanga duplicates Kigali, Rubavu duplicates Musanze), leaving 10 locations.

**Outputs** (written to the Drive project folder, never to GitHub): one CSV per location
in `raw/`, the combined table `rwanda_weather_daily.csv`, and `rwanda_weather_raw.zip`
for the public submission folder.

## Setup

The next cell mounts Google Drive (in Colab), fetches the code from GitHub, installs the
dependencies and makes the `src` package importable. Run locally, it simply uses the
repository the notebook was opened from.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Flaviaa4/rwanda-rain-prediction.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Drive holds the data, models and results so that they survive runtime disconnects.
    from google.colab import drive
    drive.mount("/content/drive")

    # The code always comes from GitHub: clone it the first time, update it afterwards.
    # A folder without src/ is a leftover clone of an empty repository, so it is replaced.
    REPO_DIR = Path("/content/rwanda-rain-prediction")
    if (REPO_DIR / "src").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    requirements = REPO_DIR / "requirements.txt"
    if not requirements.exists():
        raise FileNotFoundError(f"{requirements} is missing: push the code to {REPO_URL} first.")

    # pip's messages are captured and shown only on failure, so the cell stays quiet when
    # Colab already provides every package.
    install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                              str(requirements)], capture_output=True, text=True)
    if install.returncode:
        print(install.stdout[-3000:])
        print(install.stderr[-3000:])
        raise RuntimeError("pip install failed; see the messages above.")
else:
    # Locally the notebook is opened from the notebooks/ folder of the repository.
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)

In [ ]:
from src.config import load_config, resolve_paths
from src.data_download import build_raw_dataset

cfg = load_config()
paths = resolve_paths(cfg)
print("Project folder:", paths.project_dir)

## Download and assemble

The download sends one request per location and saves each file as soon as it arrives,
so if the runtime disconnects you can simply run the cell again: finished locations are
skipped. Expect a few minutes in total. The summary printed at the end should show
164,360 rows (10 locations x 16,436 days) and missing values only in `CLOUD_AMT` and
`ALLSKY_SFC_SW_DWN` (about 6.7%, the years 1981 to 1983).

In [ ]:
raw = build_raw_dataset(cfg, paths)
raw.head()

## Next step

Upload `rwanda_weather_raw.zip` from the Drive project folder to the public submission
folder, then open `02_rain_prediction.ipynb`.